In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import hashlib
from IPython.display import display, Image
root = Path('.').resolve()
output_root = root / 'Figures/Generated'
manifest = {'data': 'Data/^GSPC/Market/df_Indicators.csv', 'cutoff': '2025-04-02', 'observations': 21, 'ticker': '^GSPC'}
output = output_root / 'TechnicalCharts'
output.mkdir(parents=True, exist_ok=True)
data = root / manifest['data']
frame = pd.read_csv(data, index_col='Date', parse_dates=True).loc[:manifest['cutoff']].tail(manifest['observations'])
assert len(frame) == 21 and frame.index[-1].strftime('%Y-%m-%d') == '2025-04-02'
display(frame.tail())

def candlestick(axis, x_values, open_values, high_values, low_values, close_values, width=0.6):
    x_values = np.asarray(x_values)
    open_values = np.asarray(open_values, dtype=float)
    high_values = np.asarray(high_values, dtype=float)
    low_values = np.asarray(low_values, dtype=float)
    close_values = np.asarray(close_values, dtype=float)
    upward = close_values >= open_values
    body_bottom = np.minimum(open_values, close_values)
    body_height = np.maximum(np.abs(close_values - open_values), 0.02)
    for mask, color in ((upward, 'green'), (~upward, 'red')):
        if mask.any():
            axis.vlines(x_values[mask], low_values[mask], high_values[mask], color=color, lw=1.5, zorder=3)
            axis.bar(x_values[mask], body_height[mask], bottom=body_bottom[mask], width=width, color=color, edgecolor=color, lw=0.5, zorder=3)
    axis.plot([], [], color='green', lw=6, label='Up Day')
    axis.plot([], [], color='red', lw=6, label='Down Day')

def _format_panel(axis, x_values, ticks, tick_labels, ylabel):
    axis.set_ylabel(ylabel, color='black', fontsize=20, fontweight='bold')
    axis.yaxis.get_offset_text().set_fontsize(20)
    axis.yaxis.get_offset_text().set_fontweight('bold')
    axis.tick_params(axis='y', labelsize=20, width=1.5)
    for label in axis.get_yticklabels():
        label.set_fontweight('bold')
    axis.set_xlabel('Date', fontsize=20, fontweight='bold')
    axis.set_xticks(ticks)
    axis.set_xticklabels(tick_labels, rotation=0, ha='center')
    axis.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in axis.get_xticklabels():
        label.set_fontweight('bold')
    axis.set_xlim([x_values[0] - pd.Timedelta(days=2), x_values[-1] + pd.Timedelta(days=2)])
    axis.grid(True, linestyle='--', alpha=0.3)

def _format_legend(axis, location='upper left'):
    handles, labels = axis.get_legend_handles_labels()
    if handles:
        axis.legend(handles, labels, loc=location, fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
        axis.get_legend().get_frame().set_edgecolor('black')
        axis.get_legend().get_frame().set_linewidth(2)

def plot_panel_1(axis, window, x_values, ticks, tick_labels):
    base_value = window['close'].iloc[0]
    open_values = window['open'].values / base_value * 100
    high_values = window['high'].values / base_value * 100
    low_values = window['low'].values / base_value * 100
    close_values = window['close'].values / base_value * 100
    axis.plot(x_values, window['bb_lower'].values / base_value * 100, label='BB Lower', color='red', lw=4, linestyle=':')
    axis.plot(x_values, window['bb_upper'].values / base_value * 100, label='BB Upper', color='green', lw=4, linestyle=':')
    axis.fill_between(x_values, window['bb_upper'].values / base_value * 100, window['bb_lower'].values / base_value * 100, alpha=0.1, color='red', label='BB Band')
    candlestick(axis, x_values, open_values, high_values, low_values, close_values)
    axis.plot(x_values, window['sma200'].values / base_value * 100, label='SMA200', lw=4, linestyle='--', color='purple')
    axis.axhline(100, color='black', linestyle='-', lw=1.5, alpha=0.25)
    _format_panel(axis, x_values, ticks, tick_labels, 'Price (Start = 100)')
    _format_legend(axis)

def plot_panel_2(axis, window, x_values, ticks, tick_labels):
    axis.plot(x_values, window['cmf_20d'].values, label='CMF (20d)', color='blue', lw=3)
    axis.fill_between(x_values, 0, window['cmf_20d'].values, alpha=0.15, color='blue')
    axis.axhline(0, color='black', linestyle='-', lw=2, alpha=0.3)
    _format_panel(axis, x_values, ticks, tick_labels, 'CMF')
    _format_legend(axis)
    secondary_axis = axis.twinx()
    secondary_axis.plot(x_values, window['atr_14d_norm'].values, label='Normalized ATR (14d)', color='red', lw=3)
    _format_panel(secondary_axis, x_values, ticks, tick_labels, 'Normalized ATR')
    _format_legend(secondary_axis, 'upper right')

def plot_panel_3(axis, window, x_values, ticks, tick_labels):
    colors = ['green' if value >= 0 else 'red' for value in window['macd_hist_12_26_9_norm']]
    axis.bar(x_values, window['macd_hist_12_26_9_norm'].values, alpha=0.3, label='Normalized MACD Hist (12, 26, 9)', color=colors)
    axis.axhline(0, color='black', linestyle='-', lw=1.5, alpha=0.2)
    _format_panel(axis, x_values, ticks, tick_labels, 'MACD Histogram')
    _format_legend(axis)
    secondary_axis = axis.twinx()
    secondary_axis.plot(x_values, window['rsi_14d'].values, label='RSI (14d)', color='purple', lw=3)
    secondary_axis.plot(x_values, [70] * len(x_values), color='red', linestyle='--', lw=3, alpha=0.5, label='Overbought (70)')
    secondary_axis.plot(x_values, [30] * len(x_values), color='green', linestyle='--', lw=3, alpha=0.5, label='Oversold (30)')
    secondary_axis.set_ylim(-5, 105)
    _format_panel(secondary_axis, x_values, ticks, tick_labels, 'RSI')
    _format_legend(secondary_axis, 'upper right')

def plot_panel_4(axis, window, x_values, ticks, tick_labels):
    axis.plot(x_values, window['stoch_k'].values, label='Stochastic K', color='blue', lw=3)
    axis.plot([x_values[0], x_values[-1]], [80, 80], color='red', linestyle='--', lw=3, alpha=0.6, label='Overbought (80)')
    axis.plot([x_values[0], x_values[-1]], [20, 20], color='green', linestyle='--', lw=3, alpha=0.6, label='Oversold (20)')
    axis.set_ylim(-5, 105)
    _format_panel(axis, x_values, ticks, tick_labels, 'Stochastic K')
    _format_legend(axis)
    secondary_axis = axis.twinx()
    secondary_axis.plot(x_values, window['adx'].values, label='ADX', color='black', lw=3)
    secondary_axis.plot([x_values[0], x_values[-1]], [25, 25], color='grey', linestyle=':', lw=3, alpha=0.6, label='ADX Trending (25)')
    _format_panel(secondary_axis, x_values, ticks, tick_labels, 'ADX')
    _format_legend(secondary_axis, 'upper right')

def plot_panel_5(axis, window, x_values, ticks, tick_labels):
    axis.plot(x_values, window['hv_10d_ann'].values, label='HV (10d, annualized)', lw=3, color='blue')
    axis.plot(x_values, window['hv_126d_ann'].values, label='HV (126d, annualized)', lw=3, color='black')
    _format_panel(axis, x_values, ticks, tick_labels, 'Annualized Volatility')
    _format_legend(axis)
    secondary_axis = axis.twinx()
    secondary_axis.plot(x_values, window['log_return_1d_q05_roll126d'].values, label='5% VaR (126d)', color='orange', lw=3)
    secondary_axis.plot(x_values, window['es_5pct_126'].values, label='5% ES (126d)', color='red', lw=3, linestyle='--')
    secondary_axis.axhline(0, color='black', linestyle='-', lw=1.5, alpha=0.3)
    _format_panel(secondary_axis, x_values, ticks, tick_labels, 'Tail Risk (Log Return)')
    _format_legend(secondary_axis, 'upper right')

def plot_panel_6(axis, window, x_values, ticks, tick_labels):
    axis.plot(x_values, window['skewness_126'].values, label='Skewness (126d)', color='black', lw=3)
    axis.axhline(0, color='grey', linestyle='--', lw=3, alpha=0.5, label='Skewness = 0')
    _format_panel(axis, x_values, ticks, tick_labels, 'Skewness')
    _format_legend(axis)
    secondary_axis = axis.twinx()
    secondary_axis.plot(x_values, window['kurtosis_126'].values, label='Excess Kurtosis (126d)', color='orange', lw=3, linestyle='--')
    secondary_axis.axhline(0, color='lightcoral', linestyle='-', lw=3, alpha=0.5, label='Excess Kurtosis = 0')
    _format_panel(secondary_axis, x_values, ticks, tick_labels, 'Excess Kurtosis')
    _format_legend(secondary_axis, 'upper right')

def plot_panel_7(axis, window, x_values, ticks, tick_labels):
    axis.plot(x_values, window['dd_20'].values, label='Drawdown (20d)', lw=3, color='red')
    axis.plot(x_values, window['ulcer_index'].values, label='Ulcer Index', lw=3, color='black', linestyle='--')
    axis.axhline(0, color='black', linestyle='-', lw=2, alpha=0.3)
    _format_panel(axis, x_values, ticks, tick_labels, 'Drawdown / Pain (%)')
    _format_legend(axis)

def create_chart(window, ticker, date_text, panels, chart_index, save_figure_dir, transparent=False):
    figure, axes = plt.subplots(len(panels), 1, figsize=(40, 7 * len(panels)), sharex=False)
    axes = axes if isinstance(axes, np.ndarray) else [axes]
    x_values = window.index
    ticks = window.index[::max(1, len(window) // 10)]
    tick_labels = [date.strftime('%Y-%m-%d') for date in ticks]
    for index, panel_function in enumerate(panels):
        panel_function(axes[index], window, x_values, ticks, tick_labels)
    chart_folder = Path(save_figure_dir) / f'Chart {chart_index}'
    chart_folder.mkdir(parents=True, exist_ok=True)
    figure_path = chart_folder / f'{ticker}_{date_text}_chart{chart_index}.png'
    figure.suptitle(f"{ticker} - {date_text.replace('-', '')}", fontsize=28, fontweight='bold', y=0.995)
    figure.tight_layout(rect=[0, 0.03, 1, 0.95])
    figure.savefig(figure_path, transparent=transparent, dpi=144)
    plt.close(figure)
    return figure_path

def create_technical_chart(window, ticker, date_text, save_figure_dir):
    panel_requirements = {plot_panel_1: ['open', 'high', 'low', 'close', 'sma200', 'bb_lower', 'bb_upper'], plot_panel_2: ['cmf_20d', 'atr_14d_norm'], plot_panel_3: ['macd_hist_12_26_9_norm', 'rsi_14d'], plot_panel_4: ['stoch_k', 'adx'], plot_panel_5: ['hv_10d_ann', 'hv_126d_ann', 'log_return_1d_q05_roll126d', 'es_5pct_126'], plot_panel_6: ['skewness_126', 'kurtosis_126'], plot_panel_7: ['dd_20', 'ulcer_index']}
    panel_functions = [plot_panel_1, plot_panel_2, plot_panel_3, plot_panel_4, plot_panel_5, plot_panel_6, plot_panel_7]
    missing = {f'panel {index + 1}': [column for column in panel_requirements[panel] if column not in window.columns] for index, panel in enumerate(panel_functions)}
    missing = {panel: columns for panel, columns in missing.items() if columns}
    if missing:
        raise KeyError(f'Missing columns required by the 7 panels: {missing}')
    panels_per_chart = [2, 2, 3]
    if sum(panels_per_chart) != len(panel_functions):
        raise ValueError('Panel layout must account for all chart panels.')
    charts = []
    start = 0
    for chart_index, panel_count in enumerate(panels_per_chart, start=1):
        end = start + panel_count
        charts.append(create_chart(window, ticker, date_text, panel_functions[start:end], chart_index, save_figure_dir))
        start = end
    return tuple(charts)
paths = create_technical_chart(frame, manifest['ticker'], manifest['cutoff'], output)
for path in paths:
    display(Image(filename=str(path)))
import shutil
for path, name in zip(paths, ['Figure 02 Market Charts.png', 'Figure 03 Risk Charts.png', 'Figure 04 Volume Charts.png']):
    shutil.copyfile(path, output_root / name)
